## RAG step by step


**RAG 구조**
```
문서
 ↓
Chunking
 ↓
Embedding
 ↓
Vector DB
 ↓
사용자 질문
 ↓
질문 Embedding
 ↓
유사 문서 검색 Retrieval
 ↓
검색 결과 + 질문을 LLM에게 전달
 ↓
답변 Generation
```


**추후 확장**
Retrieve → Synthesize → Verify → Grounding

## 1. Chunking

문서를 검색하기 좋게 나눈 하나의 텍스트 조각

In [1]:
with open("data/school.txt", "r", encoding="utf-8") as f:
    text = f.read()

chunks = [
    chunk.strip()
    for chunk in text.split("\n\n")
    if chunk.strip()
]

for chunk in chunks:
    print(chunk)

학교 도서관은 오전 9시부터 오후 9시까지 운영한다.
도서 대출 기간은 최대 14일이다.
학생은 한 번에 최대 5권까지 대출할 수 있다.
졸업 요건은 총 130학점 이상 이수하는 것이다.
학생식당은 오전 11시부터 오후 7시까지 운영한다.


## 2. Embedding

문장을 숫자 벡터로 바꾸기

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embeddings = model.encode(chunks)

print(embeddings.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5088.65it/s]


(5, 384)


## 3. 질문 Embedding

In [4]:
question = "책은 며칠까지 빌릴 수 있어?"

question_embedding = model.encode(question)

## 4. 가장 비슷한 문서 찾기

In [5]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

In [6]:
scores = []

for chunk, embedding in zip(chunks, embeddings):
    score = cosine_similarity(question_embedding, embedding)
    scores.append((score, chunk))

scores.sort(reverse=True)

for score, chunk in scores:
    print(score, chunk)

0.50994813 도서 대출 기간은 최대 14일이다.
0.45492673 학생은 한 번에 최대 5권까지 대출할 수 있다.
0.23444241 학교 도서관은 오전 9시부터 오후 9시까지 운영한다.
0.16224743 학생식당은 오전 11시부터 오후 7시까지 운영한다.
0.12169331 졸업 요건은 총 130학점 이상 이수하는 것이다.


## 5. Top-K Retrieval

보통 유사도가 비슷한거 하나가 아니라 여러개를 가져옴

In [7]:
top_k = 3

retrieved = scores[:top_k]

for score, chunk in retrieved:
    print(chunk)

도서 대출 기간은 최대 14일이다.
학생은 한 번에 최대 5권까지 대출할 수 있다.
학교 도서관은 오전 9시부터 오후 9시까지 운영한다.


## 6. LLM에 Context 넣기

검색된 문서를 LLM에게 넘김. 

In [8]:
context = "\n".join([
    chunk
    for score, chunk in retrieved
])

prompt = f"""
아래 참고 문서만 사용해서 질문에 답하세요.

[참고 문서]
{context}

[질문]
{question}

[답변]
"""

print(prompt)


아래 참고 문서만 사용해서 질문에 답하세요.

[참고 문서]
도서 대출 기간은 최대 14일이다.
학생은 한 번에 최대 5권까지 대출할 수 있다.
학교 도서관은 오전 9시부터 오후 9시까지 운영한다.

[질문]
책은 며칠까지 빌릴 수 있어?

[답변]

